# Butadiene Optimization

:::{div}
:class: colab-badge
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blinkletter/UsingPsi4v2/blob/main/ch4/4C-butadiene/simple_butadiene.ipynb)
:::

(conformers-of-butadiene-simple)=
## Conformers of Butadiene
We expect butadiene to prefer a **flat** *syn* or *anti* conformation. These **two** possible **conformations** are designated "*s-cis*" and "*s-trans*". The "s" refers to the &sigma; bond involved. 

:::{figure} images/cistrans-butadiene.png
:label: fig-butadiene-scan
:width: 40%
:align: center
:alt: cis and trans butadiene structures

The two stable conformations of 1,3-butadiene.
:::

We understand that the effect of conjugation provides stabilization that greatly favours a flat conformation. In this notebook we will describe the structure of butadiene using a Z-matrix with a single variable that can be cahnged. We can then explore the effect of starting from different torsional angles for the central C–C bond when we optimize the structure. Is perfectly flat the preferred conformation? Let us explore.

## Setting up this Notebook

The collapsed [code cell below](#setup-cell-simple-butadiene) will **load in** all the *Python* packages and set global variables for the calculations in this notebook. If you are using *Colab*, **run this cell first**, watch it eventually crash, and then **run it again** and you will be set up.

In [48]:
#| label: setup-cell-simple-butadiene
#| tags: [remove-output, hide-input]

####################################################
### Import Libraries and set up global variables ###
####################################################

from pathlib import Path
import json

import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Install packages only if needed
# Google Colab does not include every Python package
#
# Psi4 cannot be installed with pip. On Colab we first install conda
# (condacolab), which RESTARTS the session. After the restart, run
# this cell again and Psi4 will be installed with conda.
try:                  # if the package exists import it, otherwise install it
    import psi4
except ImportError:
    if IN_COLAB:
        !pip -q install condacolab
        import condacolab
        condacolab.install()
        !mamba install -q -y -c conda-forge psi4
    import psi4

try:
    import py3Dmol
except ImportError:
    !pip -q install py3Dmol
    import py3Dmol

try:
    import skimage          # scikit-image draws the static orbital and density surfaces
except ImportError:
    !pip -q install scikit-image
    import skimage

# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Style sheets and helper functions from the book's GitHub repository
if IN_COLAB and not Path("styles/tufte.mplstyle").exists():
    !wget -q -P styles https://raw.githubusercontent.com/blinkletter/UsingPsi4v2/main/styles/tufte.mplstyle
    !wget -q -P styles https://raw.githubusercontent.com/blinkletter/UsingPsi4v2/main/styles/S2_classic2.mplstyle
    !wget -q -N https://raw.githubusercontent.com/blinkletter/UsingPsi4v2/main/psi4_helpers.py

# The methane structure file used in this chapter
if IN_COLAB and not Path("data/methane.xyz").exists():
    !wget -q -P data https://raw.githubusercontent.com/blinkletter/UsingPsi4v2/main/ch4/4C-butadiene/data/butadiene_scan.json

if IN_COLAB:
    plt.style.use("styles/tufte.mplstyle")
else:
    sys.path.insert(0, "../..")                 # psi4_helpers.py is two folders up
    plt.style.use("../../styles/tufte.mplstyle")

import psi4_helpers as ph

# Psi4 output (.out, .log, .molden, cube files, scans) goes into output/, which is not kept in git.
# Small results files that another page reads (.json, .csv) go into data/, which is kept in git.
output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

# 3D pictures: False = static images (as in the book); True = rotatable py3Dmol viewers
INTERACTIVE = True

### Setting *Psi4* Options

The code below sets options for the *Psi4* package. I set the number of threads to 2, which is the limit for *Google Colab*. Your own computer may have more processor cores, but *Colab* will only allow you the power of two for free.

In [49]:
# Resources for every calculation on this page (house limits: 4 GB, 4 threads)
psi4.set_memory("4 GB")
psi4.set_num_threads(2)

# Exact (non-density-fitted) integrals
psi4.set_options({"scf_type": "pk"})

## Create the Geometry

We must tell *Psi4* the structure of our molecule. Here I am using a Z-matrix. This is the best way to describe molecules when planning a conformational scan. In a Z-matrix we describe the molecule as a set of relative bond distances, bond angles and dihedral torsion angles. We can change these values directly and tell *Psi4* to freeze the value we are using in a scan.

### The Z-matrix

To create the Z-matrix, we first **number** the atoms. Then we declare the geometry **relative** to the **first atom**. Consider the scheme in [](#fig-zmatrix-butadiene-scan) and **compare** it to the [code that establishes the Z-matrix](#zmatrix-butadiene-simple) below. That angle for the bond torsion of the central C–C bond is defined using the **variable** `D`, that can be addressed and changed using methods in the molecule object.

:::{figure} images/Z-matrixButadiene2.png
:label: fig-zmatrix-butadiene-scan
:width: 50%
:align: center
:alt: z-matrix diagram of butadiene

A diagram of the layout of the molecular geometry for building the Z-matrix. 
:::


In [50]:
#| label: zmatrix-butadiene-simple
zmat = '''
0 1
C
C   1  1.3
C   2  1.5    1  120.0
C   3  1.3    2  120.0     1  D
H   1  1.1    2  120.0     3  0.0
H   1  1.1    5  120.0     2  180.0
H   2  1.1    1  120.0     3  180.0
H   3  1.1    4  120.0     2  180.0
H   4  1.1    3  120.0     2  0.0
H   4  1.1    9  120.0     3  180.0

D   = 180
'''

mol = psi4.geometry(zmat)


### Interpeting the Z-matrix

The [code block above](#zmatrix-butadiene-simple) set a block of text that described the structure of the molecule using a Z-matrix. 

In the Z-matrix stated above we can trace the structure out line-by-line.

- the **first line**, `0 1`, is the charge and multiplicity. Butadiene is neutral (`0`) and in a singlet state (`1`, all electrons are spin-paired).
- The following **text block** describes the **molecule**
    - the **first line**, `C`, simply states that **atom #1** is carbon.
    - the **second line**, `C 1 1.3` describes **atom #2**. It says `2` is connected to `1` and the distance is `1.3` Å. When only two atoms are known, all we have is a distance.
    - the **third line**, `C 2 1.5 1 120.0` describes **atom #3**. `3` is connected to `2` at a distance of `1.5` Å. We have three atoms now and so we could have a bond angle. Here the `3 2 1` bond angle is  120.0`°.
    - the **fourth line** describes **atom #4**. `C 3 1.3 2 120.0 1 D` means that the `4 3` distance is `1.3` Å; the `4 3 2` bond angle is `120`°; and the `4 3 2 1` dihedral (bond torsion) is set by the variable `D` (D is set to 180° later in the Z-matrix).
    - the **fifth line** describes **atom #5**, a hydrogen atom: `H 1 1.1 2 120.0 3 0.0`. It is connected to atom #1 with a bond distance of 1.1 Å. The angle between atoms `5`, `1`, and `2` is 120°. The `5 1 2 3` dihedral angle is 0°. **Compare** that with **atom #6**, with the same connection, distance and bond angle, but a **dihedral** of 180°. 
    - **Trace** the connectivity, distance, bond angle and dihedral angle for all the other atoms. Are they consistant with the diagram in [](#fig-zmatrix-butadiene-scan).

- The **next text block** is separated by a **blank line**. There must be a blank line. 
    -  Here we assign the **variable**, D. We can address that variable with `mol.mol.set_variable("D", 175)` to **set the value** of D to another value. We can also query the value from the result of a calculation with `mol.get_variable("D")`

:::{warning} All Z-Matrices are not the Same
I had used the Z-matrix for butadiene from the [NIST Computational Chemistry Comparison and Benchmark Database](https://cccbdb.nist.gov/) {cite:p}`cccbdb` and it **failed** in several calculations. Symmetry **errors** were reported. The Z-matrix in **this notebook** is one that I made myself. It is identical, except that I numbered the atoms **differently**. I do not know the reason for the **error**, but every software package has its **quirks** and it is sometimes best to just accept them and move forward. 
:::



## Optimizing the Structure

In the [code below](#optimization-butadiene-simple), I will set the `1 2 3 4` dihedral angle represented by the variable `D` in the Z-matrix. In the first calculation, I will set `D` to zero degrees. If we are using the [original Z-matrix](#zmatrix-butadiene-simple), then the molecule is perfectly flat and has $C_{2v}$ symmetry.

In [51]:
#| label: optimization-butadiene-simple

# set torsion angle
D = 0.0

# set up the structure. assign a new value for "D" and name the output file.
mol.set_variable("D", D) 
psi4.set_output_file(output_dir / f"butadiene_simple_{D:03.0f}.out", False) # filenames will be "butadiene_00X.out"

# optimize the geometry starting at a torsional angle of "D"
E = psi4.optimize("HF/6-31G", molecule = mol) 

steps = psi4.variable("OPTIMIZATION ITERATIONS")
print(f"Steps: {steps}")
print(f"Energy: {E:.7f} hartree")
torsion = mol.get_variable("D")
print(f"Optimized torsion angle: {torsion:.1f} degrees")
print(f"Symmetry: {mol.get_full_point_group()}")

Optimizer: Optimization complete!
Steps: 7.0
Energy: -154.8589540 hartree
Optimized torsion angle: -0.0 degrees
Symmetry: C2v


### Show the Molecule

The [code cell above](#optimization-butadiene-simple) optimized the geometry and reported the optimized `1 2 3 4` dihedral angle. Take note of the results: the optimized angle, the energy and the symmetry. We can visualize the structure using a helper function that calls *Py3DMol* to display the result of the optimization calculation.



In [52]:
# show the molecule using the function in psi4_helpers.py
ph.show_molecule(mol, interactive=INTERACTIVE)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## It's Your Turn

Modify the [optimization code block](#optimization-butadiene-simple) above and optimize at D = 180°. Note the the optimized angle, the energy and the symmetry. Are D = 0° and D = 180° the two minimum energies? 

Perform another calculation with D = 170°. Does the optimization return to a dihedral angle of 180°? Now try an optimization with D = 10°. Does the optimization return to a dihedral angle of 0°? 